In [8]:
!pip install imblearn ploty

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Using cached imblearn-0.0-py2.py3-none-any.whl (1.9 kB)
ERROR: Could not find a version that satisfies the requirement ploty (from versions: none)
ERROR: No matching distribution found for ploty


In [9]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [10]:
path = '/content/drive/MyDrive/Analítica'

In [11]:
# Librerias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder

In [12]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from pylab import rcParams
from imblearn.under_sampling import NearMiss
from imblearn.over_sampling import RandomOverSampler
from imblearn.combine import SMOTETomek
from imblearn.ensemble import BalancedBaggingClassifier
from collections import Counter
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, cohen_kappa_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
import plotly.express as px
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# Proyecto Final

 **Integrantes**:
- Román Pardo Alejandro
- Riubi Zuñiga Andrey
- Salazar Vega Rodrigo
- Verduzco Lozano Iván Antonio

El objetivo principal de este proyecto es 'Predecir los proveedores potencialmente fraudulentos' en función de las reclamaciones presentadas por ellos. También descubriremos cuáles son las variables más importantes que ayudan a detectar el comportamiento de proveedores potencialmente fraudulentos para que podamos señalar dichos reclamos y realizar una investigación profunda sobre ellos.

Los significados de algunas variables son:

#### 1. Inpatient and outpatient data:
- **ClaimID:** esta es una identificación única para cada reclamo enviado.
- **Bene_ID:** contiene la identificación única de los beneficiarios registrados para el plan de seguro.
- **AttendingPhysician:** columna que contiene el id de los médicos que atendieron al paciente.
- **OperatingPhysician:** la columna contiene la identificación de los médicos que operaron al paciente.
- **ClmDiagnosisCode:** contiene códigos de diagnóstico que los proveedores realizan en los pacientes.
- **ClmProcedureCode:** contiene códigos de procedimientos a los que se someten los pacientes.
- **Proveedor:** identificación única de los proveedores de atención médica.
- **InscClaimAmtReimbursed:** monto total pagado al reclamante después de la liquidación.
- **ClaimStartDt / ClaimEndDt:** estas columnas tienen la fecha en que se presentaron los reclamos y la fecha en que se liquidaron los reclamos, respectivamente.
- **AdmissionDt / HighDt:** Fecha en que el paciente ingresó en el hospital y la fecha en que el paciente fue dado de alta. Aplicable solo para datos de pacientes hospitalizados.

#### 2. Beneficiaries data:
- **Bene_ID:** contiene la identificación única de los beneficiarios registrados para el plan de seguro.
- **DOB:** Fecha de nacimiento de los beneficiarios registrados en el régimen de seguro.
- **Gender:** Género de los beneficiarios registrados en el régimen de seguro.
- **State/Country:** contiene el código de estado/país para los miembros registrados.
- **Premedical conditions:** hay algunas columnas como RenalDiseaseIndicator, ChronicCond_Depression, ChronicCond_Diabetes, etc. para indicar si el miembro tiene alguna condición médica previa.

#### 3. Target data:
- **Provider:** esta es una identificación única para cada uno de los proveedores de atención médica.
- **Target:** esta columna tiene dos valores, Sí y No, que indican si el proveedor correspondiente está marcado como fraude potencial o no.

In [14]:
# Load Train Dataset

Train=pd.read_csv(path + "/data proyecto/Train-1542865627584.csv")
Train_Beneficiarydata=pd.read_csv(path + "/data proyecto/Train_Beneficiarydata-1542865627584.csv")
Train_Inpatientdata=pd.read_csv(path + "/data proyecto/Train_Inpatientdata-1542865627584.csv")
Train_Outpatientdata=pd.read_csv(path + "/data proyecto/Train_Outpatientdata-1542865627584.csv")

FileNotFoundError: ignored

In [ ]:
Train_Inpatientdata

In [ ]:
Train_Outpatientdata

In [ ]:
Train_Beneficiarydata

In [ ]:
Train

In [ ]:
unique_ben_inpat = Train_Inpatientdata['BeneID'].unique()
unique_ben_outpat = Train_Outpatientdata['BeneID'].unique()
valores_comunes = set(unique_ben_inpat) & set(unique_ben_outpat)
valores_comunes_lista = list(valores_comunes)

In [ ]:
len(Train_Beneficiarydata['BeneID'].unique())

In [ ]:
len(unique_ben_inpat)

In [ ]:
len(valores_comunes_lista)

In [ ]:
len(unique_ben_outpat)

In [ ]:
plt.hist(Train_Beneficiarydata['Gender'])
plt.title('Distribución del Genero')
plt.xlabel('Genero')
plt.ylabel('Frecuencia')

In [ ]:
#Establecemos las calumnas con datos de tiempo en formato de fecha
Train_Beneficiarydata['DOD'] = pd.to_datetime(Train_Beneficiarydata['DOD'])
Train_Beneficiarydata['DOB'] = pd.to_datetime(Train_Beneficiarydata['DOB'])

Train_Outpatientdata['ClaimStartDt'] = pd.to_datetime(Train_Outpatientdata['ClaimStartDt'])
Train_Outpatientdata['ClaimEndDt'] = pd.to_datetime(Train_Outpatientdata['ClaimEndDt'])

Train_Inpatientdata['ClaimStartDt'] = pd.to_datetime(Train_Inpatientdata['ClaimStartDt'])
Train_Inpatientdata['ClaimEndDt'] = pd.to_datetime(Train_Inpatientdata['ClaimEndDt'])

In [ ]:
#Calculamos le edad de los beneficiarios, tomando en cuenta la fecha del fallecimiento más reciente
t_dod = Train_Beneficiarydata['DOD'].unique()
t_dod = [x for x in t_dod if pd.isnull(x) == False]
dod_max = max(t_dod)
Train_Beneficiarydata['DOD'].fillna(dod_max, inplace=True)
Train_Beneficiarydata['Age'] = np.round(((Train_Beneficiarydata['DOD'] - Train_Beneficiarydata['DOB']).dt.days)/365.0,0)

#Agregamos una columna para la duración de la reclamación
Train_Inpatientdata['CDT'] = (Train_Inpatientdata['ClaimEndDt'] - Train_Inpatientdata['ClaimStartDt']).dt.days
Train_Outpatientdata['CDT'] = (Train_Outpatientdata['ClaimEndDt'] - Train_Outpatientdata['ClaimStartDt']).dt.days

In [ ]:
#Agregamos una columna que indique si es ingresado o no
Train_Inpatientdata['is_inpatient'] = 1
Train_Outpatientdata['is_inpatient'] = 0

#Unimos los datos de Pacientes In y Out
Train_hd = pd.concat([Train_Inpatientdata, Train_Outpatientdata], axis=0, ignore_index=True)

#Juntamos los datos del beneficiario y del provedor para fomrar el dataset a utilizar
Train_hd = pd.merge(Train_hd, Train_Beneficiarydata, on='BeneID')
Train_hd = pd.merge(Train_hd, Train[['Provider', 'PotentialFraud']], on='Provider', how='left')

In [ ]:
Train_hd

In [ ]:
plt.hist(Train_hd.query('AttendingPhysician != "nan"')['PotentialFraud'])

In [ ]:
plt.hist(Train_hd.query('OperatingPhysician != "nan"')['PotentialFraud'])

In [ ]:
plt.hist(Train_hd.Age, edgecolor = 'white')
plt.title('Distribución de la edad')
plt.xlabel('Edad')
plt.ylabel('Frecuencia')

In [ ]:
Train_hd.isnull().sum()

In [ ]:
#Realizamos diferencia entre el deducible y el reembolsable
#Non-Hospitalization Reimbursement Deductible difference (NH_DMR)
#Hospitalization Reimbursement Deductible difference (H_DMR)

Train_hd2 = Train_hd

#Sustitutendo nulos con la moda
Train_hd2['DeductibleAmtPaid'].fillna(0, inplace=True)

#Agregamos columna con la diferencia entre lo deducible y lo reembolsable
Train_hd2['DRD'] = Train_hd2['InscClaimAmtReimbursed'] - Train_hd2['DeductibleAmtPaid']
#Train_hd2['NH_DRD'] = Train_hd2['IPAnnualReimbursementAmt'] - Train_hd2['IPAnnualDeductibleAmt']
#Train_hd2['H_DRD'] = Train_hd2['OPAnnualReimbursementAmt'] - Train_hd2['OPAnnualDeductibleAmt']

#Cuantifica los códigos de procedimeintos
Train_hd2['ClmProcedureCode_count'] = Train_hd2['ClmProcedureCode_1'].notnull().astype(int) + Train_hd2['ClmProcedureCode_2'].notnull().astype(int) + Train_hd2['ClmProcedureCode_3'].notnull().astype(int) + Train_hd2['ClmProcedureCode_4'].notnull().astype(int) + Train_hd2['ClmProcedureCode_5'].notnull().astype(int) + Train_hd2['ClmProcedureCode_6'].notnull().astype(int)
Train_hd2.drop(['ClmProcedureCode_1', 'ClmProcedureCode_2', 'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5', 'ClmProcedureCode_6'], axis=1, inplace=True)

#Cuantifica el número de códigos de diagnosticos
Train_hd2['Diagnosis_count'] = Train_hd2['ClmDiagnosisCode_1'].notnull().astype(int) + Train_hd2['ClmDiagnosisCode_2'].notnull().astype(int) + Train_hd2['ClmDiagnosisCode_3'].notnull().astype(int) + Train_hd2['ClmDiagnosisCode_4'].notnull().astype(int) + Train_hd2['ClmDiagnosisCode_5'].notnull().astype(int) + Train_hd2['ClmDiagnosisCode_6'].notnull().astype(int) + Train_hd2['ClmDiagnosisCode_7'].notnull().astype(int) + Train_hd2['ClmDiagnosisCode_8'].notnull().astype(int) + Train_hd2['ClmDiagnosisCode_9'].notnull().astype(int) + Train_hd2['ClmDiagnosisCode_10'].notnull().astype(int)
Train_hd2.drop(['ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3', 'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6', 'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9', 'ClmDiagnosisCode_10'], axis=1, inplace=True)

#Agregamos una columna que indique si el paciente sufrió un operación
Train_hd2['Operating'] = Train_hd2['OperatingPhysician'].apply(lambda val: 0 if val != val else 1)

#Modificar al codificación de la columna de enfermefdad renal
Train_hd2['RenalDiseaseIndicator'] = Train_hd2['RenalDiseaseIndicator'].apply(lambda val: 0 if val != "Y" else 1).values

In [ ]:
plt.hist(Train_hd2.query('PotentialFraud == "Yes"')['ClmProcedureCode_count'], edgecolor = 'white')

In [ ]:
plt.hist(Train_hd2.query('PotentialFraud == "Yes"')['DRD'], edgecolor = 'white')

In [ ]:
#Eliminamos columnas con formato de fecha
Train_hd2 = Train_hd2.drop(['ClaimStartDt', 'ClaimEndDt', 'DOB', 'DOD',
                            'AdmissionDt', 'DischargeDt'], axis = 1)

#Eliminamos columnas que no aportan información
Train_hd2 = Train_hd2.drop(['BeneID', 'ClaimID', 'NoOfMonths_PartACov', 'NoOfMonths_PartBCov',
    'AttendingPhysician','OperatingPhysician', 'OtherPhysician', 'ClmAdmitDiagnosisCode', 'DiagnosisGroupCode'], axis = 1)


In [ ]:
# Convert to numeric the provider ID
Train_hd2['Provider'] = Train_hd2['Provider'].apply(lambda x: x.split('PRV')[1]).astype(int)

In [ ]:
Train_hd2.isnull().sum()

In [ ]:
Train_hd2

In [ ]:
sc = MinMaxScaler()
X = Train_hd2.drop(['PotentialFraud'], axis = 1).values
Y = Train_hd2['PotentialFraud'].apply(lambda val: 0 if val != "Yes" else 1).values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
                                                    X,
                                                    Y,
                                                    test_size=0.3,
                                                    shuffle = True,
                                                    random_state = 137,
                                                    stratify = Y
                                                    )

X_train_s = sc.fit_transform(X_train)
X_test_s = sc.transform(X_test)

# X_train_s and X_test_s to DataFrame
# Get columns without target
columns = Train_hd2.columns
columns = columns.drop('PotentialFraud')
X_train_s = pd.DataFrame(X_train_s, columns = columns)
X_test_s = pd.DataFrame(X_test_s, columns = columns)

In [ ]:
X_train_aux = np.concatenate((X_train_s, y_train.reshape(-1, 1)), axis = 1)
X_train_aux = pd.DataFrame(X_train_aux, columns = Train_hd2.columns)
X_train_aux.head()

In [ ]:
corr = X_train_aux.corr()
corr_target = abs(corr['PotentialFraud'])
relevant_features = corr_target[corr_target > 0.05]
relevant_features

In [ ]:
# Get columns of revelevant features
relevant_features = list(relevant_features.index)
# relevant_features except PotentialFraud
relevant_features.remove('PotentialFraud')
relevant_features.append('Provider')
relevant_features

In [ ]:
# Get columns of irrelevant features
irrelevant_features = list(set(columns) - set(relevant_features))
irrelevant_features

In [ ]:
mat_cov = np.cov(X_train_s[irrelevant_features].T)
eig_val, eig_vec = np.linalg.eig(mat_cov)
var_exp = [(i / sum(eig_val)) for i in sorted(eig_val, reverse = True)]
var_exp_cum = np.cumsum(var_exp)

plt.bar(range(1, len(eig_val) + 1), var_exp, alpha = 0.5, align = 'center')
plt.step(range(1, len(eig_val) + 1), var_exp_cum, where = 'mid')
plt.xlabel('No. Componentes')
plt.ylabel('Varianza explicada')
plt.show

In [ ]:
pca = PCA(n_components=7)
components = pca.fit_transform(X_train_s[irrelevant_features])
components_test = pca.transform(X_test_s[irrelevant_features])

total_var = pca.explained_variance_ratio_.sum() * 100

fig = px.scatter_3d(
    components, x=0, y=1, z=2, color=y_train,
    title=f'Total Explained Variance: {total_var:.2f}%',
    labels={'0': 'PC 1', '1': 'PC 2', '2': 'PC 3'}
)
fig.show()

In [ ]:
# X_train_filtered
X_train_filtered = X_train_s[relevant_features]
X_test_filtered = X_test_s[relevant_features]
X_train_filtered.columns

In [ ]:
# Concatenamos las columnas numéricas con las columnas nominales transformadas
train_nom = pd.DataFrame(components, columns = [str(x) for x in range(0,7)])
test_nom = pd.DataFrame(components_test, columns = [str(x) for x in range(0,7)])

X_train_filtered = pd.concat([X_train_s, train_nom], axis=1)
X_test_filtered = pd.concat([X_test_s, test_nom], axis=1)

In [ ]:
def metricas(real, predict):
    # Calculo de sensitivity y specificity
    res = list()
    prec,recall,_,_ = precision_recall_fscore_support(real,
                                                      predict,
                                                      pos_label=True,average=None)

    fpr, tpr, thresholds = metrics.roc_curve(real, predict, pos_label=1)
    auc = metrics.auc(fpr, tpr)
    res.append([recall[0],recall[1],accuracy_score(real,predict),cohen_kappa_score(real, predict),auc, f1_score(real, predict)])
    resumen = pd.DataFrame(res,columns = ['sensitivity','specificity', 'Accuracy val', 'Kappa Value:', 'Auc', 'F1_score:'])

    print(resumen)
    # display confusion_matrix
    cm = confusion_matrix(real, predict, labels=np.unique(real))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=np.unique(real))
    disp.plot()

In [ ]:
fig, ax = plt.subplots(nrows=1, figsize=(8, 4))
sns.histplot(y_train, ax=ax)
plt.title('Distribución antes del submuestreo')
_ = plt.xlabel('Etiquetas')

In [ ]:
# Codigo del submuestreo
nm = NearMiss(version= 1, n_neighbors=5)
X_train_sub, y_train_sub = nm.fit_resample(X_train_filtered, y_train)
X_test_sub, y_test_sub = nm.fit_resample(X_test_filtered, y_test)

print('Antes del submuestreo de y: ', Counter(y_train))
print('Muestras submuestradas de y: ', Counter(y_train_sub))

print(type(y_test_sub))
print('Antes del submuestreo de test: ', Counter(y_test))
print('Despues del submuestreo de test: ', Counter(y_test_sub))
# print('Muestras submuestradas de test: ', Counter(X_test_sub))

In [ ]:
# Del X_train, sacamos conjunto de validación
X_train, X_val, y_train, y_val = train_test_split(X_train_sub,
                                                    y_train_sub,
                                                    test_size=0.2,
                                                    shuffle = True,
                                                    random_state = 137,
                                                    stratify = y_train_sub
                                                    )

In [ ]:
# Cross validation
def cross_val(model, X_train, y_train, cv):
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    return np.mean(scores)

In [ ]:
# Buscando el mejor k con cross validation
rango = range(2, 15)
scores = []

for k in tqdm(rango):
    knn = KNeighborsClassifier(n_neighbors=k)
    scores.append(cross_val(knn, X_val, y_val, 8))

plt.plot(rango, scores)

In [ ]:
# Knn
knn = KNeighborsClassifier(n_neighbors = 9)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test_sub)
metricas(y_test_sub, y_pred)

In [ ]:
# Buscando los mejores estimadores random forest
estimators = [10, 50, 75, 100]
scores = []

for e in tqdm(estimators):
    rf = RandomForestClassifier(n_estimators=e, random_state=137)
    scores.append(cross_val(rf, X_val, y_val, 5))

plt.plot(estimators, scores)

In [ ]:
clf = RandomForestClassifier(n_estimators=90)
clf.fit(X_train, y_train)
prediccion = clf.predict(X_test_sub)

metricas(y_test_sub, prediccion)

In [ ]:
# Cross validation con Regresión Logística
lr = LogisticRegression(random_state=137)
scores = cross_val_score(lr, X_val, y_val, cv=5, scoring='accuracy')
print(np.mean(scores))

In [ ]:
# Cross validation con Multi Layer Perceptron
solvers = ['sgd', 'adam']

for s in tqdm(solvers):
    mlp = MLPClassifier(solver=s, random_state=137, max_iter=1000)
    scores = cross_val_score(mlp, X_val, y_val, cv=5, scoring='accuracy')
    print(f'{s}: {np.mean(scores)}')